DH演習2026 · 教材 第48版 · 2026-r48 · 2026-11-12

<details><summary>教材の版と相談用情報</summary>

相談時は次の情報を伝えてください。

```text
DH演習2026 · 教材 第48版
教材ID: 2026-r48
教材日付: 2026-11-12
活動: authority_links
環境: Colab / Jupyter
ソース: f3b0469f0499647a3cc5e74a1a02b1b4af49853c
```

</details>

[同じ版の教材・変更点](https://nlp.lang.osaka-u.ac.jp/dh/practicum/editions.html)


# 第6回：典拠レコードをRDFとして読み、同じ人物の記録をたどる

上から順にセルを実行してください（Shift+Enter）。値を変えたら、そのセルから下を再実行します。

資料：服部之総『黒船来航』本文第4段落（[TEI-EAJ aozora_tei](https://github.com/TEI-EAJ/aozora_tei/blob/77a675fc2771936f9544505d922d4cd45075338c/data/complete/tei_lib_lv4/50362_tei.xml)、CC0）、[Web NDL Authorities](https://id.ndl.go.jp/auth/ndla/)、[Wikidata](https://www.wikidata.org/)（CC0）

## 1. 本文と二つの候補を読む

本文の「ペリー」は、どの人物でしょうか。候補は二人います。名前だけでは区別できないので、年代と活動を比べます。本文の1848年は周囲の時代の説明で、ペリーの活動の年ではありません。

In [ ]:
import pandas as pd

passage = "一八四八年といえばブルジョア革命の波が西欧を襲った年であり、産業革命によって蒸汽船が実用化され、鉄道が実用化される時代であった。ペリーはアメリカ海軍で、世界に先がけて、最新の技術をもって従来の伝統にこだわることなく蒸汽海軍をつくった。"
print(passage)
pd.DataFrame([
    dict(候補="Matthew Calbraith Perry", 年代="1794〜1858年。1853年に日本へ遠征", 活動="米海軍の軍人。蒸気動力の艦船の推進"),
    dict(候補="Oliver Hazard Perry", 年代="1785〜1819年", 活動="米海軍の軍人。1813年のエリー湖の戦い"),
])

## 2. NDLの典拠レコードをRDFとして取得する

典拠レコードのURIの末尾に`.ttl`を付けると、ページの代わりにデータ（Turtle形式のRDF）を取得できます。`authority`を変えると、ほかの典拠レコードで試せます。ノートブックはRDFを読むライブラリrdflibを使います。

In [ ]:
authority = "00452560"

In [ ]:
#@title 操作：NDLからRDFを取得して読み込む
!pip -q install rdflib
import requests
from rdflib import BNode, Graph, Literal, Namespace, URIRef
from rdflib.namespace import FOAF, OWL, RDF, RDFS, SKOS

record = URIRef("http://id.ndl.go.jp/auth/ndlna/" + authority)
response = requests.get(f"https://id.ndl.go.jp/auth/ndlna/{authority}.ttl", headers={"User-Agent": "dh-enshu-2026-colab/1.0"})
response.raise_for_status()
g = Graph().parse(data=response.text, format="turtle")
g.bind("record", "http://id.ndl.go.jp/auth/ndlna/")
g.bind("person", "http://id.ndl.go.jp/auth/entity/")
print("トリプルの数：", len(g))

## 3. トリプルを表にする

RDFは、一つの事実を主語・述語・目的語の三つ組（トリプル）で表します。`short()`は、長いURIを`foaf:name`のような短い書き方に直します。`record:`は典拠レコード、`person:`は人物のURIの省略です。`_:`で始まる主語は、URIを持たない節点（空白ノード）です。

表を見る前に予想してください。主語は何種類あるでしょうか。

In [ ]:
def short(term):
    if isinstance(term, URIRef):
        try:
            return g.qname(term)
        except Exception:
            return str(term)
    return term.n3() if isinstance(term, BNode) else str(term)

table = pd.DataFrame([dict(主語=short(s), 述語=short(p), 目的語=short(o)) for s, p, o in g]).sort_values(["主語", "述語"])
print("主語の種類：", table["主語"].nunique())
table

## 4. レコードと人物を区別する

NDLは、典拠レコード（記録）と人物を別のURIで表します。`foaf:primaryTopic`は「このレコードの主題」を指します。生没年は人物の側、異表記はレコードの側にあります。

本文の「ペリー」は、異表記に含まれていますか。生没年は、1節の候補のどちらと一致しますか。

In [ ]:
XL = Namespace("http://www.w3.org/2008/05/skos-xl#")
RDA = Namespace("http://RDVocab.info/ElementsGr2/")

person = g.value(record, FOAF.primaryTopic)
variants = sorted(str(form) for node in g.objects(record, XL.altLabel) for form in g.objects(node, XL.literalForm))
print("レコード：", record)
print("人物：", person)
print("異表記：", variants)
print("生年：", g.value(person, RDA.dateOfBirth), "没年：", g.value(person, RDA.dateOfDeath))

## 5. ほかの機関の記録へたどる

`skos:exactMatch`は、ほかの機関の同じ内容の記録を指します。Wikidataの項目は、さらに多くの機関の識別子を記録しています。次のセルは、Wikidataの検索サービスへSPARQLの問合せを送り、`item`の項目が持つ外部の識別子を一覧にします。

互いを参照する記録は、独立した証拠とは限りません。`dct:source`で、このレコードが何を根拠に作られたかを確かめてください。

In [ ]:
item = "Q61855"

def wikidata(query):
    response = requests.get("https://query.wikidata.org/sparql", params={"query": query, "format": "json"}, headers={"User-Agent": "dh-enshu-2026-colab/1.0"})
    response.raise_for_status()
    return pd.DataFrame([{key: value["value"] for key, value in row.items()} for row in response.json()["results"]["bindings"]])

print("NDLが指す記録：", sorted(str(o) for o in g.objects(record, SKOS.exactMatch)))
print("NDLの根拠：", sorted(str(o) for o in g.objects(record, URIRef("http://purl.org/dc/terms/source"))))
identifiers = wikidata('''
SELECT ?propertyLabel ?id WHERE {
  wd:''' + item + ''' ?claim ?id .
  ?property wikibase:directClaim ?claim ;
            wikibase:propertyType wikibase:ExternalId .
  SERVICE wikibase:label { bd:serviceParam wikibase:language "ja,en". }
}
ORDER BY ?propertyLabel
''')
print("Wikidataの項目が持つ外部の識別子：", len(identifiers), "件")
identifiers[identifiers["id"].isin([authority, "37133702", "n50009036"])]

## 6. 条件で候補を探す

名前で一人ずつ調べる代わりに、条件に合う人物をまとめて探せます。次の問合せは、アメリカ海軍に所属し、NDLの典拠IDを持つ人物を生年順に並べます。`wdt:P31 wd:Q5`は「人間である」、`wdt:P241`は「所属する軍」、`wdt:P349`は「NDLの典拠ID」です。

`name`を含む人物は何人いますか。Oliver Hazard Perryは、この一覧にいません。一覧にいない人物を、候補から外せますか。

In [ ]:
name = "ペリー"
branch = "Q11220"

people = wikidata('''
SELECT ?person ?personLabel ?birth ?death ?ndl WHERE {
  ?person wdt:P31 wd:Q5 ;
          wdt:P241 wd:''' + branch + ''' ;
          wdt:P349 ?ndl .
  OPTIONAL { ?person wdt:P569 ?birth . }
  OPTIONAL { ?person wdt:P570 ?death . }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "ja,en". }
}
ORDER BY ?birth
''')
print("条件に合う人物：", len(people), "人")
people[people["personLabel"].str.contains(name)][["personLabel", "birth", "death", "ndl", "person"]]

## 7. 判断をトリプルで書く

`accepted`に採用するURIを入れると、ノートブックは作品内の人物`document.xml#perry`とそのURIを結ぶトリプルを書きます。述語は、URIが何を指すかで決まります。

- 人物についての記録を指すURI：`foaf:isPrimaryTopicOf`（この人物を主題とする記録）
- 人物そのものを指すURI：`owl:sameAs`（同じ対象）

実行前に予想してください。`accepted`を4節の「人物」のURIに変えると、述語はどう変わるでしょうか。

In [ ]:
accepted = "https://id.ndl.go.jp/auth/ndlna/00452560"

names_person = "/entity/" in accepted
local = URIRef("document.xml#perry")
links = Graph()
links.bind("foaf", FOAF)
links.bind("owl", OWL)
links.add((local, RDF.type, FOAF.Person))
links.add((local, RDFS.label, Literal("ペリー", lang="ja")))
links.add((local, OWL.sameAs if names_person else FOAF.isPrimaryTopicOf, URIRef(accepted)))
print(links.serialize(format="turtle"))

## 確認問題

6節の`name`を「ハイネ」に変えて実行すると、ペリーの遠征に同行した画家ウィルヘルム・ハイネが見つかります。表の`ndl`の番号を2節の`authority`に入れて4節まで再実行し、`answer`に人物のURIと生年を入れてください。

In [ ]:
answer = dict(person="", birth="")

In [ ]:
#@title 答えを確かめる
if not answer["person"] or not answer["birth"]:
    print("結果：不合格（`answer`に人物のURIと生年を入れてください）")
elif "/ndlna/" in answer["person"]:
    print("結果：不合格（典拠レコードのURIです。`foaf:primaryTopic`が指す人物のURIを入れてください）")
elif answer["person"].replace("https://", "http://") != "http://id.ndl.go.jp/auth/entity/00442917":
    print("結果：不合格（`authority`を変えて4節まで再実行し、「人物：」の行を確かめてください）")
elif str(answer["birth"]) != "1827":
    print("結果：不合格（4節の「生年：」の行を確かめてください）")
else:
    print("結果：合格")